# cvenv — component-by-component setup demo

One cell per component (`install` + `verify`). Use this as a template: copy the
single cell you need into any tutorial notebook. The GPU components
(`pytorch3d`, `mast3r`, `sam2`) need a GPU runtime; `science` runs anywhere.

> If a step changes **numpy**, restart the kernel/runtime once before continuing.

In [1]:
# Install cvenv itself (no heavy deps). Pin a tag for reproducibility.
!pip install -q "git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.2"

import cvenv
print("cvenv", cvenv.__version__)
cvenv.PlatformManager().detect_platform()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
cvenv 0.1.2


('Colab', '/content/')

In [2]:
from google.colab import drive; drive.mount('/content/drive')   # mount FIRST so the wheel persists

Mounted at /content/drive


In [3]:
import cvenv
cvenv.get_component("pytorch3d").install(from_source=True)

▶️  pytorch3d: installing…
$ /usr/bin/python3 -m pip install numpy>=2.0,<2.1
$ /usr/bin/python3 -m pip install iopath
Building PyTorch3D from source (this can take several minutes)…
$ /usr/bin/python3 -m pip install --root-user-action ignore ninja
$ /usr/bin/python3 -m pip wheel --no-deps --no-build-isolation git+https://github.com/facebookresearch/pytorch3d.git@stable -w /content/drive/MyDrive/cvenv_wheels

💾 saved reusable wheel: /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
$ /usr/bin/python3 -m pip install --force-reinstall --no-deps /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
   ↻ next time, skip the build with:
       cvenv.get_component("pytorch3d").install(wheel_url="/content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl")
     or:  cvenv install pytorch3d --wheel-url /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl


In [4]:
cvenv.get_component("pytorch3d").verify()   # want: ✅ pytorch3d … (_C OK)

✅ pytorch3d 0.7.9 (_C OK)


True

In [ ]:
from google.colab import drive; drive.mount('/content/drive')   # mount FIRST so the wheel persists

import cvenv
cvenv.get_component("pytorch3d").install(from_source=True)       # builds vs torch 2.11/cu128 → 💾 saves to Drive → installs
cvenv.get_component("pytorch3d").verify()                        # should print ✅ (_C OK)

In [5]:
# See what's available (with the teaching notes)
for c in cvenv.list_components():
    print(f"{c.name:<10} {c.summary}")
    print(f"           why: {c.teaching_note}\n")

mast3r     NAVER MASt3R matcher: clone repo, install deps, fetch checkpoint.
           why: MASt3R is used as a source checkout, not a pip package — the repo root must be on sys.path for 'import mast3r'. Its checkpoint is multi-GB, so the download resumes on failure. A local copy can be reused via checkpoint_dir= to skip the download.

opengl     PyOpenGL + system GL/GLUT dev libraries (for pyrender / rendering).
           why: Offscreen GL on a headless server needs a context: set PYOPENGL_PLATFORM=egl (GPU, fast) or osmesa (CPU, robust). The apt libs here (freeglut3-dev, libglew-dev, libsdl2-dev) only install on Debian/Ubuntu images; they're skipped elsewhere.

pytorch3d  Facebook PyTorch3D (differentiable 3D). Wheel if possible, else source build.
           why: The single hardest install here. A CUDA-matched prebuilt wheel is far faster than a source build. Always test 'import pytorch3d._C' — plain 'import pytorch3d' succeeds even when the compiled _C extension is broken. A whee

## science — base numpy-2 scientific stack
Runs anywhere. This is all you need for pure-numpy/scipy material (Kalman filters, Lie groups).

In [6]:
cvenv.get_component("science").install()
cvenv.get_component("science").verify()

✅ science: already installed — skipping.
✅ science: numpy 2.0.2, scipy + cv2 4.13.0 import OK


True

## pytorch3d — differentiable 3D (GPU)
Fastest with a prebuilt wheel matching this runtime's torch/CUDA/cp version.
Set `MY_WHEEL` to that URL, or leave `None` to try the official index / source build.

In [6]:
MY_WHEEL = "https://www.dropbox.com/scl/fi/6ec1i5k5t5antughl5voy/pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl?rlkey=fvdc14gdgsrn2zrj5nnrli0y9&dl=1"
cvenv.get_component("pytorch3d").install(wheel_url=MY_WHEEL)
cvenv.get_component("pytorch3d").verify()   # tests import pytorch3d._C

▶️  pytorch3d: installing…
$ /usr/bin/python3 -m pip install numpy>=2.0,<2.1
$ /usr/bin/python3 -m pip install iopath
⬇️  downloading wheel: pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
Installing PyTorch3D from wheel: pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
$ /usr/bin/python3 -m pip install --force-reinstall --no-deps /tmp/pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl


RuntimeError: the provided wheel installed but 'import pytorch3d._C' failed:
    ImportError: /usr/local/lib/python3.12/dist-packages/pytorch3d/_C.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib
Runtime: python=cp312, torch=2.11.0+cu128, cuda=12.8
Wheel:   pytorch3d-0.7.9-cp312-cp312-linux_x86_64.whl
Most likely the wheel's torch/CUDA build doesn't match this runtime. Rebuild the wheel against the runtime's torch/CUDA, or pass from_source=True to build here.

## mast3r — NAVER matcher (GPU)
Clones the repo, installs deps, downloads the checkpoint. Pass `checkpoint_dir=` to reuse a local copy.

In [ ]:
cvenv.get_component("mast3r").install()   # add checkpoint_dir="..." to skip the download
cvenv.get_component("mast3r").verify()

## sam2 — Meta Segment Anything 2 (GPU)
pip-installs SAM2 and downloads its checkpoint. Store it under a persistent path via `checkpoint_dir=`.

In [ ]:
cvenv.get_component("sam2").install(checkpoint_dir="checkpoints")
cvenv.get_component("sam2").verify(checkpoint_dir="checkpoints")